# 365 Probabilidades — Dia #044
## Qual a probabilidade de a sua procrastinação não ter nada a ver com preguiça?

**Tipo:** Comportamental
**Data de publicação:** 2026-07-27
**Ferramenta:** Python
**Decisão analisada:** Devo me cobrar mais para parar de procrastinar, ou existe outro caminho?
**Hashtag:** #365Probabilidades #Dia044

---

### 📖 A História

Ontem eu escrevi, aqui mesmo, que faz semanas que eu não tenho coragem de abrir o Strava. Que o meu treinador continua colocando os treinos no TrainingPeaks, e eu finjo que não é comigo.

Chamei aquilo de falta de coragem. Mas existe um nome mais preciso: procrastinação.

E durante muito tempo eu achei que procrastinar era um defeito de caráter. Preguiça. Indisciplina. Uma pessoa organizada como eu não deveria fazer isso, então quando eu fazia, eu me cobrava. Me chamava de mole, de fraca.

Só que a ciência diz uma coisa quase ofensiva de tão contraintuitiva. Procrastinar não tem quase nada a ver com preguiça. E se cobrar, que parece a solução óbvia, é justamente o que mantém você presa no ciclo.

---

### 📚 O Conceito: Não É Preguiça, É Regulação de Emoção

Tim Pychyl e Fuschia Sirois viraram a mesa da pesquisa sobre procrastinação com uma ideia simples: procrastinar não é um problema de gestão de tempo. É um problema de gestão de emoção.

Quando uma tarefa dispara algo desconfortável, tédio, medo de falhar, ansiedade, tristeza, o cérebro faz uma troca silenciosa: alivia o mal-estar agora, adiando, e empurra o custo para o seu eu futuro. É reparo de humor de curto prazo. Você não está sendo preguiçosa. Você está fugindo de um sentimento.

E a meta-análise mais completa da área confirma isso pelo avesso. Steel (2007), reunindo 216 estudos, encontrou uma correlação praticamente nula entre procrastinar e inteligência: r = 0,03. Gente brilhante procrastina igual. O que prevê a procrastinação é o quanto a tarefa incomoda e o quanto a impulsividade fala mais alto. Emoção e autorregulação, não capacidade.

---

### 🧮 O Modelo

Uso os correlatos da meta-análise de Steel para mostrar de onde a procrastinação realmente vem, comparo estatisticamente o peso do incômodo da tarefa com o peso da inteligência, e trago o experimento de Wohl sobre o que quebra o ciclo.

**Fontes:**
- Steel, P. (2007). *The Nature of Procrastination*. **Psychological Bulletin**, 133(1), 65-94.
  Meta-análise: 216 estudos, 691 correlações independentes.
- Sirois, F. & Pychyl, T. (2013). *Procrastination and the Priority of Short-Term Mood Regulation*. **Social and Personality Psychology Compass**. (conceito)
- Wohl, M., Pychyl, T. & Bennett, S. (2010). *I forgive myself, now I can study*. **Personality and Individual Differences**, 48, 803-808. N=119 calouros.

**Nota metodológica:** os números são correlações e tamanhos de efeito de meta-análise e de estudo longitudinal, não proporções de survey autorrelatado. Por isso o fator ×0.80 não se aplica. Os IC 95% das correlações usam um N agregado conservador **declarado** (500 por correlato) apenas para fins ilustrativos, já que a meta-análise reporta o número de estudos (K), não o N de cada correlato.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# Paleta 365 Probabilidades
DOURADO = '#c8a84b'
VERMELHO = '#c0392b'
VERDE    = '#1a5f5a'
CINZA    = '#6b6a64'

np.random.seed(44)  # Dia #044

print("✅ Bibliotecas carregadas")


In [ ]:
# --- DADOS DA LITERATURA ---
# Steel (2007), Psychological Bulletin — meta-análise (216 estudos, 691 correlações)
# Correlação de cada traço com PROCRASTINAR (sinal preservado)

correlatos = {
    'Baixa conscienciosidade': -0.62,   # menos conscienciosidade -> mais procrastina
    'Distração':                0.45,
    'Impulsividade':            0.41,
    'Aversão à tarefa':         0.40,
    'Baixa autoeficácia':      -0.38,
    'Neuroticismo':             0.24,
    'Inteligência':             0.03,    # ~ZERO: não é questão de capacidade
}
N_estudos = 216
N_correlacoes = 691

# N agregado conservador DECLARADO para IC ilustrativo (Steel reporta K, não N por correlato)
N_ic = 500

# Wohl, Pychyl & Bennett (2010) — N=119 calouros, duas provas
N_wohl = 119
beta_interacao = 0.23   # p=.02 — quanto mais autoperdão, menos procrastinação depois
b_afeto = 0.24          # p=.009 — mediado por afeto negativo

print("=" * 66)
print("  DADOS DA LITERATURA — PROCRASTINAÇÃO")
print("=" * 66)
print(f"  Steel 2007 · {N_estudos} estudos · {N_correlacoes} correlações")
print("  Correlação de cada traço com procrastinar:")
for nome, r in sorted(correlatos.items(), key=lambda x: -abs(x[1])):
    print(f"    {nome:<26} r = {r:+.2f}")
print()
print(f"  Wohl 2010 · N={N_wohl} calouros:")
print(f"    Autoperdão -> menos procrastinação depois   (β={beta_interacao}, p=.02)")
print(f"    Efeito mediado por afeto negativo           (b={b_afeto}, p=.009)")
print("=" * 66)


In [ ]:
# --- O MODELO: o incômodo da tarefa pesa mais que a inteligência? ---

def fisher_ci(r, n, alpha=0.05):
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    zc = 1.959964
    lo, hi = z - zc*se, z + zc*se
    return np.tanh(lo), np.tanh(hi)

r_intel = correlatos['Inteligência']
r_task  = correlatos['Aversão à tarefa']

ci_intel = fisher_ci(r_intel, N_ic)
ci_task  = fisher_ci(r_task,  N_ic)

# Monte Carlo — P(|vínculo com a tarefa| > |vínculo com a inteligência|)
n_sim = 100_000
se = 1 / np.sqrt(N_ic - 3)
z_intel = np.random.normal(np.arctanh(r_intel), se, n_sim)
z_task  = np.random.normal(np.arctanh(r_task),  se, n_sim)
p_tarefa_pesa_mais = np.mean(np.abs(np.tanh(z_task)) > np.abs(np.tanh(z_intel)))
p_str = ">99.9" if p_tarefa_pesa_mais > 0.999 else f"{p_tarefa_pesa_mais*100:.1f}"

print("=" * 66)
print("  MODELO — DE ONDE VEM A PROCRASTINAÇÃO")
print("=" * 66)
print(f"  Vínculo com INTELIGÊNCIA:     r = {r_intel:+.2f}  IC 95% [{ci_intel[0]:+.2f}, {ci_intel[1]:+.2f}]")
print(f"    -> o IC cruza o zero: estatisticamente, ~nada")
print(f"  Vínculo com AVERSÃO À TAREFA: r = {r_task:+.2f}  IC 95% [{ci_task[0]:+.2f}, {ci_task[1]:+.2f}]")
print(f"    -> longe do zero: o quanto a tarefa te incomoda importa")
print()
print(f"  Monte Carlo ({n_sim:,} simulações):")
print(f"  → P(o incômodo da tarefa pesar mais que a inteligência): {p_str}%")
print()
print(f"  RESPOSTA:")
print(f"  → A probabilidade de a sua procrastinação ter mais a ver")
print(f"    com o que você SENTE do que com a sua inteligência: {p_str}%")
print()
print(f"  O QUE QUEBRA O CICLO (Wohl 2010):")
print(f"  → Não é se cobrar. É se perdoar.")
print(f"    Quem se perdoou por procrastinar procrastinou MENOS na")
print(f"    vez seguinte (β={beta_interacao}), porque o perdão desarma")
print(f"    o afeto negativo que alimenta o adiamento (b={b_afeto}).")
print("=" * 66)


In [ ]:
# --- VISUALIZAÇÃO — 3 GRÁFICOS SEPARADOS ---

# ── GRÁFICO 1 — Não é preguiça, não é burrice ──
fig1, ax1 = plt.subplots(figsize=(10, 6))
cats = ['Inteligência', 'Aversão\nà tarefa', 'Impulsividade']
vals = [abs(correlatos['Inteligência']), abs(correlatos['Aversão à tarefa']), abs(correlatos['Impulsividade'])]
cores = [CINZA, DOURADO, VERMELHO]
bars = ax1.bar(cats, vals, color=cores, alpha=0.9, width=0.5)
ax1.set_ylabel('Força da ligação com procrastinar  (|r|)')
ax1.set_ylim(0, 0.5)
ax1.set_title('Procrastinar não é falta de inteligência\nMeta-análise Steel 2007 — 216 estudos, 691 correlações',
              fontsize=13, pad=15)
for bar, v in zip(bars, vals):
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.012, f'{v:.2f}',
             ha='center', fontweight='bold', fontsize=16)
ax1.text(0, 0.09, 'quase\nzero', ha='center', fontsize=11, color=CINZA, style='italic')
plt.figtext(0.5, 0.005,
            'Inteligência r=0,03 · o que pesa é como a tarefa faz você se sentir | Steel 2007 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-044-grafico-01-nao-e-preguica.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 — ASSINATURA ESTATÍSTICA: forest plot dos correlatos ──
fig2, ax2 = plt.subplots(figsize=(10, 6.5))
itens = sorted(correlatos.items(), key=lambda x: abs(x[1]))
nomes = [k for k, _ in itens]
rs = [abs(v) for _, v in itens]
ys = np.arange(len(nomes))
for i, (nome, r) in enumerate(itens):
    lo, hi = fisher_ci(abs(r), N_ic)
    cor = CINZA if nome == 'Inteligência' else (VERDE if r > 0 else DOURADO)
    ax2.plot([lo, hi], [i, i], color=cor, lw=2.5, alpha=0.8, zorder=2)
    ax2.scatter([abs(r)], [i], color=cor, s=90, zorder=3, edgecolor='#333')
    ax2.text(abs(r)+0.03, i+0.18, f'r={r:+.2f}', fontsize=10, color='#333')
ax2.axvline(0, color=VERMELHO, ls='--', lw=1.5, alpha=0.7)
ax2.set_yticks(ys); ax2.set_yticklabels(nomes)
ax2.set_xlabel("Correlação com procrastinar  |r|  (IC 95%, Fisher z, N declarado = 500)")
ax2.set_xlim(-0.1, 0.85)
ax2.set_title("A ASSINATURA ESTATÍSTICA\nO IC da inteligência encosta no zero; o das emoções não",
              fontsize=13, pad=15)
plt.figtext(0.5, 0.005,
            'Forest plot dos correlatos · Steel 2007 (216 estudos) | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-044-grafico-02-assinatura.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 — O que quebra o ciclo: perdão, não cobrança ──
fig3, ax3 = plt.subplots(figsize=(10, 6))
x = np.array([0, 1])          # baixo -> alto autoperdão
y = np.array([0.80, 0.42])    # procrastinação futura (esquemático, direção do efeito)
ax3.plot(x, y, color=VERDE, lw=3, marker='o', markersize=12, zorder=3)
ax3.fill_between(x, y, 0, color=VERDE, alpha=0.08)
ax3.set_xticks([0, 1]); ax3.set_xticklabels(['Se cobrar\n(culpa)', 'Se perdoar\n(autocompaixão)'])
ax3.set_yticks([])
ax3.set_ylim(0, 1)
ax3.set_ylabel('Procrastinação na próxima vez')
ax3.set_title('O que quebra o ciclo: perdão, não cobrança\nWohl, Pychyl & Bennett 2010 — N=119 calouros',
              fontsize=13, pad=15)
ax3.annotate('quem se perdoou\nprocrastinou MENOS\nna prova seguinte',
             xy=(0.92, 0.44), xytext=(0.22, 0.80), fontsize=11, color=VERDE, fontweight='bold',
             ha='left', va='top',
             arrowprops=dict(arrowstyle='->', color=VERDE, lw=2))
plt.figtext(0.5, 0.005,
            'Esquema da direção do efeito · interação β=0,23 (p=.02), mediada por afeto negativo | Wohl 2010 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-044-grafico-03-perdao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")


### 💡 O Insight

**A correlação entre procrastinar e inteligência é 0,03. Praticamente zero.**

Gente brilhante adia igual. O que a meta-análise de Steel mostra é que a procrastinação não mora na sua capacidade, e sim em quanto a tarefa te incomoda e quanto a impulsividade fala mais alto no momento. A probabilidade de o seu adiamento ter mais a ver com o que você sente do que com quão capaz você é passa de 99%.

Ou seja: quando você adia, não é porque é preguiçosa. É porque a tarefa dispara um sentimento ruim, e adiar é o jeito mais rápido de fazer esse sentimento sumir por alguns minutos. Procrastinação é regulação de emoção disfarçada de falha de caráter.

E é aqui que o dado fica bonito. A saída não é se cobrar mais. É o contrário. Wohl acompanhou calouros em duas provas: quem se **perdoou** por ter procrastinado na primeira procrastinou **menos** na segunda. Porque a culpa alimenta o afeto negativo, e o afeto negativo alimenta o próximo adiamento. O perdão corta esse fio.

Se cobrar parece disciplina. Mas, nos números, é gasolina no ciclo que você quer apagar.

*Qual tarefa você tem adiado não por preguiça, mas porque ela mexe com algo que você ainda não quis sentir?*

---

### ⚠️ Limitações do Modelo

- As correlações são meta-analíticas (Steel 2007 reporta K de estudos, não o N de cada correlato); os IC 95% usam um N conservador declarado (500) apenas para ilustrar a incerteza.
- Correlação não é causa: os correlatos descrevem com o que a procrastinação anda junto, não uma cadeia causal única.
- Wohl 2010 tem N=119, amostra estudantil; o efeito do autoperdão é uma interação mediada, não uma prescrição universal.
- O Gráfico 3 é esquemático: mostra a direção do efeito de Wohl, não valores brutos do estudo.
- O fator ×0.80 não foi aplicado: são tamanhos de efeito, não proporções de survey autorrelatado.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
